In [1]:
# ============================================================
# RELATÓRIO DE PASSIVO — envio por E-MAIL + WHATSAPP (GARE11 e GAME11)
# ============================================================
import os
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
from datetime import datetime
from dotenv import load_dotenv

# Helpers Z-API já usados pelo relatório ATIVO — importados, não duplicados.
from resumo_ativo import enviar_texto_whatsapp, enviar_documento_whatsapp

load_dotenv()

# Grupo "Guardian - PASSIVO inteligência listados" (formato: <id>-group).
GRUPO_PASSIVO_PHONE = os.getenv("GRUPO_PASSIVO_PHONE")

In [2]:
# ============================================================
# 1) DATA E CAMINHOS DOS PDFs — único lugar onde a data é editada
# ============================================================
data = "16.10.2026"   # ou: datetime.today().strftime("%d.%m.%Y")

DIARIOS = (
    r"g:\.shortcut-targets-by-id\1YZLLTFOuLniE6hO7WNHXl2c3MSa4TOv6\X. GAME11"
    r"\GUARDIAN Institucional\5. Planejamento & Processos\2. 2023\4. Research\2025"
    r"\5. Inteligência Passivos Guardian\2. RELATÓRIOS\1. DIARIOS"
)

# Atenção: o GARE exporta "Passivos_" (plural) e o GAME "Passivo_" (singular).
pdf_gare = rf"{DIARIOS}\GARE\Passivos_GARE11_{data}.pdf"
pdf_game = rf"{DIARIOS}\GAME\Passivo_GAME11_{data}.pdf"

In [3]:
# ============================================================
# 2) TEXTOS DO DIA ✍️ — a ÚNICA célula que muda todo dia
# ------------------------------------------------------------
#   CORPO_*  -> miolo HTML do e-mail (só as tags <p>; o invólucro
#               <html><body> e a assinatura entram em _html_email, célula 3).
#               Em HTML, use &gt; e &lt; no lugar de > e < (ex.: &gt;15M).
#   WHATS_*  -> texto do WhatsApp, enviado VERBATIM (emojis e "> " normais).
# ============================================================

# ---------------------------- GARE11 ----------------------------
CORPO_GARE = """
<p>Olá, time,</p>

<p>Resumo do cenário diário GARE11 — <b>16/10/2026</b>: cota em queda de <b>-1,60%</b> acompanhando o IFIX Tijolo (-0,3%), fechando a <b>R$ 8,62</b>, com a Patria vendendo R$ 1,4 mi na faixa &gt;15M e o UBS vendendo R$ 195k na faixa 3M-15M. Relatório completo em anexo.</p>

<p><b>📊 Desempenho do Dia</b><br>
GARE11: -1,60% | IFIX Tijolo: -0,3% | Daytrade: 44% do volume total</p>

<p><b>📈 Compras</b> (TOP 10 – R$ 635k)<br>
5k-20k: 25,73% | 20k-50k: 19,88% | 1k-5k: 18,66%</p>

<p>Nenhum institucional ficou entre os 5 maiores compradores. Entraram 7 novos cotistas no dia, que somaram R$ 479k em compras. Um cotista que zerou a posição ontem voltou a comprar: R$ 121k, entrando na faixa 100k-500k. Outro, da faixa 50k-100k, comprou R$ 56k e migrou para 100k-500k.</p>

<p><b>📉 Vendas</b> (TOP 10 – R$ 2,5 mi)<br>
&gt;15M: 33,38% | 100k-500k: 17,95% | 50k-100k: 12,32%</p>

<p>A Patria Renda Urbana (R$ 816k, 13º dia consecutivo) e a Patria Recebíveis Imobiliários (R$ 551k, 24º dia consecutivo) seguem vendendo na faixa &gt;15M e somaram R$ 1,4 mi. O UBS AG, London Branch vendeu R$ 195k e se manteve na faixa 3M-15M. O Morgan Stanley Uruguay vendeu R$ 185k e migrou de 100k-500k para 50k-100k. Um cotista da faixa 100k-500k vendeu R$ 173k e se manteve na faixa.</p>
"""

WHATS_GARE = """📅 GARE11 - 16/10/2026

⚠️ XP E GENIAL LIDERAM SALDOS NEGATIVOS EM DIA DE QUEDA DA COTA ACOMPANHANDO O IFIX TIJOLO, COM PATRIA VENDENDO R$ 1,4 MI NA FAIXA >15M E UBS VENDENDO R$ 195K NA FAIXA 3M-15M

📈 Volume negociado foi de R$ 7,3 milhões. A cota fechou o dia em queda de -1,60%, fechando a R$ 8,62. O fundo registrou mais um dia de entrada líquida de cotistas (+550), ultrapassando o patamar de 569 mil

📊 Desempenho do Dia:

* GARE11: -1,60%
* IFIX TIJOLO: -0,3%
* Daytrade: 44% do volume total

📈 Compras (TOP 10 – R$ 635k)

* 5k-20k: 25,73%
* 20k-50k: 19,88%
* 1k-5k: 18,66%

🔹7 novos cotistas no dia, somando R$ 479k em compras.
🔹Cotista que zerou a posição ontem volta a comprar R$ 121k, entrando na faixa 100k-500k.
🔹Cotista da faixa 50k-100k compra R$ 56k, migrando para 100k-500k.

📉 Vendas (TOP 10 – R$ 2,5 mi)

* >15M: 33,38%
* 100k-500k: 17,95%
* 50k-100k: 12,32%

🔹PATRIA RENDA URBANA (R$ 816k, 13º dia consecutivo) e PATRIA RECEBIVEIS IMOBILIARIOS (R$ 551k, 24º dia consecutivo) seguem vendendo na faixa >15M, somando R$ 1,4 mi.
🔹UBS AG, LONDON BRANCH vende R$ 195k mantendo-se na faixa 3M-15M.
🔹MORGAN STANLEY URUGUAY vende R$ 185k, migrando para 50k-100k.
🔹Cotista da faixa 100k-500k vende R$ 173k mantendo-se na faixa."""


# ---------------------------- GAME11 ----------------------------
# Mesma estrutura do GARE11 — preencher com os números do dia.
CORPO_GAME = """
<p>Olá, time,</p>

<p>Resumo do cenário diário GAME11 — <b>16/10/2026</b>: cota em queda de <b>-2,85%</b> acompanhando o IFIX Papel (-0,5%), fechando a <b>R$ 8,51</b>, com o FII RBR Rendimento vendendo R$ 161k na faixa 3M-15M e o Inter Hedge comprando R$ 85k e migrando para 500k-1M. Relatório completo em anexo.</p>

<p><b>📊 Desempenho do Dia</b><br>
GAME11: -2,85% | IFIX Papel: -0,5% | Daytrade: 12% do volume total</p>

<p><b>📈 Compras</b> (TOP 10 – R$ 184k)<br>
100k-500k: 42,41% | &gt;15M: 31,81% | 5k-20k: 10,88%</p>

<p>O Inter Hedge Fundo de Investimento comprou R$ 85k (6º dia consecutivo) e migrou de 100k-500k para 500k-1M. O Guardian Strategy FII de RL comprou R$ 64k (3º dia consecutivo) e se manteve na faixa &gt;15M. Nenhum novo cotista entre os 10 maiores compradores. Um cotista da faixa 5k-20k comprou R$ 17k e migrou para 20k-50k.</p>

<p><b>📉 Vendas</b> (TOP 10 – R$ 198k)<br>
3M-15M: 80,30% | 100k-500k: 10,60% | 1k-5k: 4,98%</p>

<p>O Fdo Inv Imob - FII RBR Rendimento vendeu R$ 161k (3º dia consecutivo) e se manteve na faixa 3M-15M. A XTX Markets Trading Limited vendeu R$ 3k (6º dia consecutivo) e migrou de 50k-100k para 20k-50k. Um cotista da faixa 100k-500k vendeu R$ 21k (4º dia consecutivo) e se manteve na faixa. A Rigel Cove LP zerou novamente a posição na faixa 1k-5k, com venda de R$ 1k.</p>
"""

WHATS_GAME = """📅 GAME11 - 16/10/2026

⚠️ XP E BTG PACTUAL LIDERAM SALDOS NEGATIVOS EM DIA DE QUEDA DA COTA ACOMPANHANDO O IFIX PAPEL, COM RBR RENDIMENTO VENDENDO R$ 161K NA FAIXA 3M-15M E INTER HEDGE COMPRANDO R$ 85K, MIGRANDO PARA 500K-1M

📈 Volume negociado foi de R$ 230 mil. A cota fechou o dia em queda de -2,85%, fechando a R$ 8,51. O fundo registrou saída líquida de cotistas (-6), mantendo-se acima do patamar de 20 mil

📊 Desempenho do Dia:

* GAME11: -2,85%
* IFIX PAPEL: -0,5%
* Daytrade: 12% do volume total

📈 Compras (TOP 10 – R$ 184k)

* 100k-500k: 42,41%
* >15M: 31,81%
* 5k-20k: 10,88%

🔹INTER HEDGE FUNDO DE INVESTIMENTO compra R$ 85k (6º dia consecutivo), migrando para 500k-1M.
🔹GUARDIAN STRATEGY FII DE RL compra R$ 64k (3º dia consecutivo) mantendo-se na faixa >15M.
🔹Cotista da faixa 5k-20k compra R$ 17k, migrando para 20k-50k.

📉 Vendas (TOP 10 – R$ 198k)

* 3M-15M: 80,30%
* 100k-500k: 10,60%
* 1k-5k: 4,98%

🔹FDO INV IMOB - FII RBR RENDIMENTO vende R$ 161k (3º dia consecutivo) mantendo-se na faixa 3M-15M.
🔹XTX MARKETS TRADING LIMITED vende R$ 3k (6º dia consecutivo), migrando para 20k-50k.
🔹Cotista da faixa 100k-500k vende R$ 21k (4º dia consecutivo) mantendo-se na faixa.
🔹RIGEL COVE LP zera novamente a posição na faixa 1k-5k com venda de R$ 1k."""

In [ ]:
# ============================================================
# 3) FUNÇÕES DE ENVIO — uma implementação, usada pelos dois fundos
# ============================================================
REMETENTE = "lc@guardian-asset.com"

DESTINATARIOS = [
    "lc@guardian-asset.com", "pmk@guardian-asset.com", "msf@guardian-asset.com",
    "cm@guardian-asset.com", "lb@guardian-asset.com", "ja@guardian-asset.com",
    "gsa@guardian-asset.com", "gvm@guardian-asset.com", "rt@guardian-asset.com",
    "acl@guardian-asset.com", "lt@guardian-asset.com", "tfa@guardian-asset.com", "ffs@guardian-asset.com", "mer@guardian-asset.com"
]

# Fecha todo e-mail, nos dois fundos (anexada por _html_email).
ASSINATURA = """<p>Att,<br>

Lucas Cavalcante.</p>"""


def _html_email(corpo):
    """Embrulha o miolo HTML no mesmo estilo usado no relatório ATIVO."""
    return (
        '<html><body style="font-family: Arial, sans-serif; font-size: 14px; '
        'color: #222222; line-height: 1.55;">'
        f"{corpo.strip()}"
        f"{ASSINATURA}"
        "</body></html>"
    )


def _enviar_email(pdf_path, corpo, assunto):
    msg = MIMEMultipart()
    msg["From"] = REMETENTE
    msg["To"] = ", ".join(DESTINATARIOS)
    msg["Subject"] = assunto
    msg.attach(MIMEText(_html_email(corpo), "html"))

    # Anexar o PDF
    try:
        with open(pdf_path, "rb") as attachment:
            part = MIMEBase("application", "octet-stream")
            part.set_payload(attachment.read())
        encoders.encode_base64(part)
        part.add_header(
            "Content-Disposition",
            f"attachment; filename= {os.path.basename(pdf_path)}",
        )
        msg.attach(part)
    except Exception as e:
        print(f"❌ Falha ao anexar o arquivo: {e}")
        return

    # Enviar via SMTP do Gmail
    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as smtp_server:
            smtp_server.login(REMETENTE, os.getenv("senha_gmail"))
            smtp_server.sendmail(REMETENTE, DESTINATARIOS, msg.as_string())
        print(f"✅ E-mail enviado — {assunto}")
    except Exception as e:
        print(f"❌ Falha ao enviar e-mail: {e}")


def _enviar_whatsapp(texto, pdf_path):
    """Envia o texto e, em seguida, o PDF ao grupo do PASSIVO."""
    if not GRUPO_PASSIVO_PHONE:
        raise ValueError("GRUPO_PASSIVO_PHONE não definido no .env.")
    enviar_texto_whatsapp(texto, phone=GRUPO_PASSIVO_PHONE)
    enviar_documento_whatsapp(pdf_path, phone=GRUPO_PASSIVO_PHONE)


def enviar_passivo(fundo, pdf_path, corpo, texto_whats, data, email=True, whatsapp=True):
    """fundo: 'GARE' ou 'GAME'. Assunto: RELATÓRIO DE PASSIVO {fundo} DIA {data}."""
    if email:
        _enviar_email(pdf_path, corpo, f"RELATÓRIO DE PASSIVO {fundo} DIA {data}")
    if whatsapp:
        _enviar_whatsapp(texto_whats, pdf_path)

In [5]:
# ============================================================
# 4) 👀 PRÉ-VISUALIZAÇÃO — não envia nada. Rode antes das células 5 e 6.
# ============================================================
from IPython.display import HTML, display

for fundo, pdf_path, corpo, texto in [
    ("GARE", pdf_gare, CORPO_GARE, WHATS_GARE),
    ("GAME", pdf_game, CORPO_GAME, WHATS_GAME),
]:
    print("=" * 78)
    print(f"Assunto : RELATÓRIO DE PASSIVO {fundo} DIA {data}")
    print(f"Anexo   : {'✅' if os.path.exists(pdf_path) else '❌ NÃO ENCONTRADO'}  {pdf_path}")
    print(f"Grupo   : {GRUPO_PASSIVO_PHONE}")
    print("-" * 78)
    display(HTML(_html_email(corpo)))
    print("--- WhatsApp ".ljust(78, "-"))
    print(texto)
    print()

Assunto : RELATÓRIO DE PASSIVO GARE DIA 16.10.2026
Anexo   : ✅  g:\.shortcut-targets-by-id\1YZLLTFOuLniE6hO7WNHXl2c3MSa4TOv6\X. GAME11\GUARDIAN Institucional\5. Planejamento & Processos\2. 2023\4. Research\2025\5. Inteligência Passivos Guardian\2. RELATÓRIOS\1. DIARIOS\GARE\Passivos_GARE11_16.10.2026.pdf
Grupo   : 120363370142403605-group
------------------------------------------------------------------------------


--- WhatsApp -----------------------------------------------------------------
📅 GARE11 - 16/10/2026

⚠️ XP E GENIAL LIDERAM SALDOS NEGATIVOS EM DIA DE QUEDA DA COTA ACOMPANHANDO O IFIX TIJOLO, COM PATRIA VENDENDO R$ 1,4 MI NA FAIXA >15M E UBS VENDENDO R$ 195K NA FAIXA 3M-15M

📈 Volume negociado foi de R$ 7,3 milhões. A cota fechou o dia em queda de -1,60%, fechando a R$ 8,62. O fundo registrou mais um dia de entrada líquida de cotistas (+550), ultrapassando o patamar de 569 mil

📊 Desempenho do Dia:

* GARE11: -1,60%
* IFIX TIJOLO: -0,3%
* Daytrade: 44% do volume total

📈 Compras (TOP 10 – R$ 635k)

* 5k-20k: 25,73%
* 20k-50k: 19,88%
* 1k-5k: 18,66%

🔹7 novos cotistas no dia, somando R$ 479k em compras.
🔹Cotista que zerou a posição ontem volta a comprar R$ 121k, entrando na faixa 100k-500k.
🔹Cotista da faixa 50k-100k compra R$ 56k, migrando para 100k-500k.

📉 Vendas (TOP 10 – R$ 2,5 mi)

* >15M: 33,38%
* 100k-500k: 17,95%
* 50k-100k: 12,32%

🔹PATRIA RENDA URBANA (R$ 816k, 13º dia cons

--- WhatsApp -----------------------------------------------------------------
📅 GAME11 - 16/10/2026

⚠️ XP E BTG PACTUAL LIDERAM SALDOS NEGATIVOS EM DIA DE QUEDA DA COTA ACOMPANHANDO O IFIX PAPEL, COM RBR RENDIMENTO VENDENDO R$ 161K NA FAIXA 3M-15M E INTER HEDGE COMPRANDO R$ 85K, MIGRANDO PARA 500K-1M

📈 Volume negociado foi de R$ 230 mil. A cota fechou o dia em queda de -2,85%, fechando a R$ 8,51. O fundo registrou saída líquida de cotistas (-6), mantendo-se acima do patamar de 20 mil

📊 Desempenho do Dia:

* GAME11: -2,85%
* IFIX PAPEL: -0,5%
* Daytrade: 12% do volume total

📈 Compras (TOP 10 – R$ 184k)

* 100k-500k: 42,41%
* >15M: 31,81%
* 5k-20k: 10,88%

🔹INTER HEDGE FUNDO DE INVESTIMENTO compra R$ 85k (6º dia consecutivo), migrando para 500k-1M.
🔹GUARDIAN STRATEGY FII DE RL compra R$ 64k (3º dia consecutivo) mantendo-se na faixa >15M.
🔹Cotista da faixa 5k-20k compra R$ 17k, migrando para 20k-50k.

📉 Vendas (TOP 10 – R$ 198k)

* 3M-15M: 80,30%
* 100k-500k: 10,60%
* 1k-5k: 4,98%

🔹

In [6]:
# ============================================================
# 5) ENVIAR E-MAILS — confira a célula 4 antes de rodar
# ============================================================
enviar_passivo("GARE", pdf_gare, CORPO_GARE, WHATS_GARE, data, whatsapp=False)
enviar_passivo("GAME", pdf_game, CORPO_GAME, WHATS_GAME, data, whatsapp=False)

✅ E-mail enviado — RELATÓRIO DE PASSIVO GARE DIA 16.10.2026
✅ E-mail enviado — RELATÓRIO DE PASSIVO GAME DIA 16.10.2026


In [7]:
# ============================================================
# 6) ENVIAR AO WHATSAPP — grupo "Guardian - PASSIVO inteligência listados"
#    4 mensagens: texto GARE -> PDF GARE -> texto GAME -> PDF GAME
# ============================================================
enviar_passivo("GARE", pdf_gare, CORPO_GARE, WHATS_GARE, data, email=False)
enviar_passivo("GAME", pdf_game, CORPO_GAME, WHATS_GAME, data, email=False)

✅ Texto enviado ao WhatsApp (120363370142403605-group). messageId: 3EB06D1D281C0868ACF46E
✅ PDF enviado ao WhatsApp (120363370142403605-group). messageId: 3EB0BDE1C82438175B59DD
✅ Texto enviado ao WhatsApp (120363370142403605-group). messageId: 3EB0CE27407DCC41442091
✅ PDF enviado ao WhatsApp (120363370142403605-group). messageId: 3EB036B4D60A43F426DD83
